# Video Crawler

In [19]:
pip install yt-dlp

In [20]:
import yt_dlp

def download_playlist(playlist_url, output_path="downloads"):
    ydl_opts = {
        'outtmpl': f'{output_path}/%(playlist_title)s/%(title)s.%(ext)s',
        'format': 'bestvideo+bestaudio/best',
        'merge_output_format': 'mp4',
        'noplaylist': False,  # đảm bảo download cả playlist
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([playlist_url])
        print("Download hoàn tất!")
    except Exception as e:
        print("Lỗi:", e)


if __name__ == "__main__":
    url = input("Nhập link playlist: ")
    download_playlist(url)

Nhập link playlist: https://www.youtube.com/playlist?list=PLoROMvodv4rOmsNzYBMe0gJY2XS8AQg16
[youtube:tab] Extracting URL: https://www.youtube.com/playlist?list=PLoROMvodv4rOmsNzYBMe0gJY2XS8AQg16
[youtube:tab] PLoROMvodv4rOmsNzYBMe0gJY2XS8AQg16: Downloading webpage
[youtube:tab] PLoROMvodv4rOmsNzYBMe0gJY2XS8AQg16: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: Stanford CS231N Deep Learning for Computer Vision I 2025
[youtube:tab] PLoROMvodv4rOmsNzYBMe0gJY2XS8AQg16 page 1: Downloading API JSON
[youtube:tab] Playlist Stanford CS231N Deep Learning for Computer Vision I 2025: Downloading 18 items of 18
[download] Downloading item 1 of 18
[youtube] Extracting URL: https://www.youtube.com/watch?v=2fq9wYslV0A
[youtube] 2fq9wYslV0A: Downloading webpage


[youtube] 2fq9wYslV0A: Downloading initial data API JSON


[youtube] 2fq9wYslV0A: Downloading android vr player API JSON


ERROR: [youtube] 2fq9wYslV0A: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Lỗi: ERROR: [youtube] 2fq9wYslV0A: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


This code attempts to move the entire `downloads` folder (which is located at `/content/downloads` by default) into the `destination_path` you provided in your Google Drive.

# Transcript Crawler

In [22]:
!pip install yt-dlp youtube-transcript-api -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 8.0 MB/s eta 0:00:00


In [23]:
import os
import re
import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound

# ============================================================
# UTILS
# ============================================================

def seconds_to_timestamp(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def sanitize_filename(name):
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.strip().replace(" ", "_")
    return name[:100]

def get_playlist_videos(playlist_url):
    """Dùng yt-dlp để lấy danh sách video_id + title từ playlist"""
    ydl_opts = {
        "quiet": True,
        "extract_flat": True,   # chỉ lấy metadata, không download video
        "skip_download": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(playlist_url, download=False)

    videos = []
    for entry in info.get("entries", []):
        if entry:
            videos.append({
                "id": entry.get("id"),
                "title": entry.get("title", f"video_{entry.get('id')}"),
                "url": f"https://www.youtube.com/watch?v={entry.get('id')}"
            })
    return videos

def get_transcript(video_id, languages=["en", "en-US", "en-GB"]):
    ytt_api = YouTubeTranscriptApi()
    try:
        # Lấy danh sách transcript available
        transcript_list = ytt_api.list(video_id)

        # Ưu tiên 1: manual transcript tiếng Anh
        try:
            transcript = transcript_list.find_manually_created_transcript(["en", "en-US", "en-GB"])
            return transcript.fetch()
        except Exception:
            pass

        # Ưu tiên 2: auto-generated tiếng Anh
        try:
            transcript = transcript_list.find_generated_transcript(["en", "en-US", "en-GB"])
            return transcript.fetch()
        except Exception:
            pass

        # Ưu tiên 3: lấy bất kỳ cái nào có
        for transcript in transcript_list:
            return transcript.fetch()

    except Exception as e:
        print(f"         ⚠️  Chi tiết lỗi: {e}")
        return None

def format_transcript(fetched_transcript):
    lines = []
    for snippet in fetched_transcript:
        timestamp = seconds_to_timestamp(snippet.start)
        lines.append(timestamp)
        lines.append(snippet.text)
        lines.append("")
    return "\n".join(lines)

# ============================================================
# MAIN PIPELINE
# ============================================================

def download_playlist_transcripts(playlist_url, output_dir="transcripts", languages=["en", "en-US", "en-GB"]):
    os.makedirs(output_dir, exist_ok=True)

    print(f"📋 Đang tải playlist metadata...\n")
    videos = get_playlist_videos(playlist_url)
    total = len(videos)
    print(f"🎬 Tìm thấy {total} video\n{'=' * 60}")

    success, skipped, failed = [], [], []

    for idx, video in enumerate(videos, start=1):
        video_id = video["id"]
        title_raw = video["title"]
        title = sanitize_filename(title_raw)
        filename = f"{title}_transcript.txt"
        filepath = os.path.join(output_dir, filename)

        print(f"[{idx}/{total}] 🎥 {title_raw}")

        if os.path.exists(filepath):
            print(f"         ⏭️  Đã tồn tại, bỏ qua\n")
            skipped.append(title)
            continue

        fetched = get_transcript(video_id, languages)
        if fetched is None:
            print(f"         ❌ Không lấy được transcript\n")
            failed.append(title_raw)
            continue

        content = format_transcript(fetched)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"Title: {title_raw}\n")
            f.write(f"URL: {video['url']}\n")
            f.write(f"Video ID: {video_id}\n")
            f.write("=" * 60 + "\n\n")
            f.write(content)

        print(f"         ✅ Lưu: {filename}\n")
        success.append(title)

    # Summary
    print("=" * 60)
    print(f"✅ Thành công : {len(success)}/{total}")
    print(f"⏭️  Bỏ qua    : {len(skipped)}/{total}")
    print(f"❌ Thất bại   : {len(failed)}/{total}")
    if failed:
        print(f"\nVideo không lấy được transcript:")
        for f in failed:
            print(f"  - {f}")
    print(f"\n📁 Output tại: /content/{output_dir}/")

# ============================================================
# RUN
# ============================================================

PLAYLIST_URL = "https://www.youtube.com/playlist?app=desktop&list=PL3FW7Lu3i5JvHM8ljYj-zLfQRF3EO8sYv"

download_playlist_transcripts(
    playlist_url=PLAYLIST_URL,
    output_dir="CS231n",
    languages=["en", "en-US", "en-GB"]
)

📋 Đang tải playlist metadata...

🎬 Tìm thấy 16 video
[1/16] 🎥 Lecture 1 | Introduction to Convolutional Neural Networks for Visual Recognition
         ⚠️  Chi tiết lỗi: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=vT1JzLTH4G4! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, plea

# Move folder

In [ ]:
import shutil
import os

source_path = '/content/downloads'

# Ensure the destination directory exists
os.makedirs(destination_path, exist_ok=True)

try:
    # Move the downloads folder to the specified destination path
    shutil.move(source_path, destination_path)
    print(f"Successfully moved '{source_path}' to '{destination_path}'")
except shutil.Error as e:
    print(f"Error moving directory: {e}")
    print("It's possible the folder already exists in the destination, or there are permission issues.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Error moving directory: [('/content/downloads/Stanford CS224N Natural Language Processing with Deep Learning I Spring 2024 I Professor Christopher Manning/Stanford CS224N： NLP with Deep Learning ｜ Spring 2024 ｜ Lecture 3 - Backpropagation, Neural Network.mp4', '/content/drive/MyDrive/DATA REF/LEARNING PATH/downloads/Stanford CS224N Natural Language Processing with Deep Learning I Spring 2024 I Professor Christopher Manning/Stanford CS224N： NLP with Deep Learning ｜ Spring 2024 ｜ Lecture 3 - Backpropagation, Neural Network.mp4', '[Errno 2] No such file or directory'), ('/content/downloads/Stanford CS224N Natural Language Processing with Deep Learning I Spring 2024 I Professor Christopher Manning/Stanford CS224N： NLP with Deep Learning ｜ Spring 2024 ｜ Lecture 15 - After DPO by Nathan Lambert.mp4', '/content/drive/MyDrive/DATA REF/LEARNING PATH/downloads/Stanford CS224N Natural Language Processing with Deep Learning I Spring 2024 I Professor Christopher Manning/Stanford CS224N： NLP with De